# Rebuttal sweeps: graph sensitivity (W1.5) & node-fraction dose response (Q1.3)

Aggregates the per-cell-type sweep results into rebuttal-style `mean ± SD` tables and dose-response plots.

- **Slide 232**: results copied verbatim from `../../../cellina/analysis/` (`results_232/`, 3 seeds) — the source of Rebuttal Tables 1 & 2.
- **Slide 210**: new sweep (`results_210/`, 5 seeds, k ∈ {10, 100, 200, 1000, 10000}, full fraction grid). Slide 210 has a third domain (`210_TVA`) that stays in training as an extra domain class but is excluded from the REF→CRC evaluation.

Workers: `graph_sensitivity/run_sensitivity.py`, `node_fraction/run_node_fraction.py`; launchers: `*/launch_per_ct.sh`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CTS = ["Fibroblast", "Endothelial", "Myeloid", "T_cell", "Epithelial"]


def load_results(root, param):
    """Collect per-run JSONs from root/<ct>/*.json into a long dataframe."""
    rows = []
    root = Path(root)
    for ct in CTS:
        for f in sorted((root / ct).glob("*.json")) if (root / ct).exists() else []:
            r = json.loads(f.read_text())
            rows.append({"cell_type": ct, param: r[param], "seed": r["seed"],
                         "pearson": r["pearson"]})
    return pd.DataFrame(rows)


def wide_table(df, param):
    """Rebuttal-style table: rows = param values, cols = cell types, mean ± SD."""
    if df.empty:
        return pd.DataFrame()
    g = df.groupby([param, "cell_type"])["pearson"].agg(["mean", "std", "count"])
    cell = g.apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f} (n={int(r['count'])})", axis=1)
    return cell.unstack("cell_type").reindex(columns=CTS)


def dose_plot(df, param, title, ax, logx=False):
    for ct in CTS:
        sub = df[df.cell_type == ct].groupby(param)["pearson"].agg(["mean", "std"])
        if sub.empty:
            continue
        ax.errorbar(sub.index, sub["mean"], yerr=sub["std"], marker="o",
                    capsize=3, label=ct)
    if logx:
        ax.set_xscale("log")
    ax.set_xlabel(param)
    ax.set_ylabel("Pearson r (top-50 DEG logFC)")
    ax.set_title(title)
    ax.legend(fontsize=8)

## W1.5 — Graph-construction sensitivity (k sweep, bandwidth = ∞)

In [ ]:
gs_232 = load_results("graph_sensitivity/results_232", "k")
gs_210 = load_results("graph_sensitivity/results_210", "k")

print("Slide 232 (3 seeds) — Rebuttal Table 1 source:")
display(wide_table(gs_232, "k"))
print("Slide 210 (5 seeds):")
display(wide_table(gs_210, "k"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
dose_plot(gs_232, "k", "Slide 232", axes[0], logx=True)
dose_plot(gs_210, "k", "Slide 210", axes[1], logx=True)
fig.suptitle("Sensitivity to neighbor-graph size k (bandwidth = ∞)")
fig.tight_layout()
fig.savefig("graph_sensitivity/graph_sensitivity_232_vs_210.png", dpi=200, bbox_inches="tight")

## Q1.3 — Node-perturbation dose response (perturbed-neighbor fraction)

In [ ]:
nf_232 = load_results("node_fraction/results_232", "fraction")
nf_210 = load_results("node_fraction/results_210", "fraction")

print("Slide 232 (3 seeds) — Rebuttal Table 2 source:")
display(wide_table(nf_232, "fraction"))
print("Slide 210 (5 seeds):")
display(wide_table(nf_210, "fraction"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
dose_plot(nf_232, "fraction", "Slide 232", axes[0])
dose_plot(nf_210, "fraction", "Slide 210", axes[1])
fig.suptitle("Dose response in the fraction of perturbed neighbors (k = 200 checkpoints)")
fig.tight_layout()
fig.savefig("node_fraction/node_fraction_232_vs_210.png", dpi=200, bbox_inches="tight")